##### Define notebook variables with widgets

Widgets is a great way to use variables in the notebook session across all cells

In [0]:
%python
dbutils.widgets.removeAll()
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container_name", "")
dbutils.widgets.text("table_name", "")

In [0]:
%python
storage_account = getArgument("storage_account")
container_name = getArgument("container_name")
table_name = getArgument("table_name","xxxx")
print (storage_account)
print (container_name)
print (table_name)

##### Access ADLS Using SAS token

Note that the recommended way to access ADLS from Databricks is by using AAD Service Principal and the backed by Azure Key Vault Databricks Secret Scope.

Here for simplicity we use SAS token.

In [0]:
%python
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account}.dfs.core.windows.net", "sp=racwdlmeop&st=2024-07-15T09:02:04Z&se=2024-08-01T17:02:04Z&spr=https&sv=2022-11-02&sr=c&sig=H4C7vXC7cDFZI8hdxZBGjrD12DYU1pNgy1RfFxeXm2I%3D")

In [0]:
%python
#List the content of ADLS folder
display(dbutils.fs.ls( f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/FlightsDelays/"))

##### Create schema and tables

After we explored the data we create schemas and tables in order to work with the data as we get used to work with regular data bases

Note that the location of a schema (database), in example below is in the default location under dbfs:/user/hive/warehouse/ and the schema directory is the name of the schema with the .db extension

However you can specify schema (database) location: `CREATE SCHEMA IF NOT EXISTS ${da.schema_name}_custom_location LOCATION '${working_dir}/${da.schema_name}.db'; `

In [0]:
show schemas

In [0]:

create schema if not exists flights

In [0]:
show schemas

In [0]:
describe schema extended flights

##### Create external table
We create a table in `flights` schema, which we created previously.

Notice using ***_header_*** option, which allows to define columns names from the header

In [0]:
use flights


###### Note! If you run in the same workspace - Please add to the table names unique prefix, example 5 digits of your ID to avoid conflicts 

In [0]:
-- Note we use header = "true"
-- create or replace table.. also works
create table if not exists  flights_delays
using csv options (
  path = 'abfss://${container_name}@${storage_account}.dfs.core.windows.net/FlightsDelays/FlightDelaysWithAirportCodes.csv',header = "true");
  

In [0]:
select * from flights_delays limit 10

In [0]:
show tables

In [0]:
describe extended flights_delays

Pay attention that this table is ***External*** and the table format is CSV. Since it's an extrernal table, the location is the original one: the Azure Data Lake Storage (ADLS) container.

Also you can see that there is no history for non-delta tables.

The history is supported by Delta format but this table is in CSV format.

Now we can work with this table as we get used working with regular tables...

In [0]:
select DepDel15, count(*) as number_of_delays
from flights_delays
group by DepDel15

Databricks visualization. Run in Databricks to view.

In [0]:
-- Note distinct counts the null values
select distinct (DepDel15)
from flights_delays

In [0]:
-- Average departure delay for each carrier
SELECT 
    Carrier, 
    AVG(CAST(DepDelay AS INT)) AS AvgDepDelay
FROM 
    flights_delays
GROUP BY 
    Carrier
ORDER BY 
    AvgDepDelay DESC;

In [0]:
-- Number of canceled flighs per month
SELECT 
    Month, 
    COUNT(*) AS CancelledFlights
FROM 
    flights_delays
WHERE 
    Cancelled = '1'
GROUP BY 
    Month
ORDER BY 
    Month;

Databricks visualization. Run in Databricks to view.

In [0]:
describe table flights_delays

What's wrong with this table defintion?

Since the original data is in CSV format, Spark can not infer precisely the correct fields types (by default). 

You can enforce schema inference also for CSV, JSON formats by setting format option : `inferSchema=true`, see below examples.

However the recomended way is to create table and define the types strictly.

In [0]:

create table if not exists flights_delays_typed (
  year int,
  month int,
  dayofmonth int,
  dayofweek int,
  carrier string,
  crsdeptime string,
  depdelay int,
  depdel15 int,
  crsarrTime string,
  arrdelay int,
  arrdel15 int,
  canceled smallint,
  originairportcode string,
  originairportname string,
  originlatitude float,
  originlongitude float,
  destairportcode string,
  destairportname string,
  destlatitude float,
  destlongitude float
) using csv options (
  path = 'abfss://${container_name}@${storage_account}.dfs.core.windows.net/FlightsDelays/FlightDelaysWithAirportCodes.csv',
  header = "true"
);

In [0]:
describe extended flights_delays_typed

##### CTEs

Spark supports CTEs - Common Table Expessions.

CTEs defines a temporary result set that can be referenced multiple times in other following CTE queries 

In order to define CTE - you use `WITH` clause

In [0]:
 -- number of delays per origin airport
 -- define CTE
 with flights_dep_delays_cte (origin_airport, dep_total_delays) as (
  select
    originairportname,
    sum(depdel15) as origindelay15min
  from
    flights_delays_typed
  group by
    originairportname
)
-- use CTE
select
  origin_airport,
  dep_total_delays
from
  flights_dep_delays_cte
order by dep_total_delays desc
limit 10

##### Create Table As Select (CTAS)

Create Delta Lake tables with CTAS

In [0]:
create
or replace table flights_delays_delta as
select
  *
from
  flights_delays

In [0]:
describe extended flights_delays_delta

Now the table type is `Managed`.  By default, if we don't specify the output format,  the tables are created in Delta format.

For managed tables, the data is copied from original location (ADLS container) to the `schema` location in dbfs (Databricks File System). 

The Databricks File System (DBFS) is a distributed file system mounted into an Azure Databricks Workspace and available on Azure Databricks clusters. 

If you delete a ***Managed*** table all its data will be deleted as well (not the case with `External` tables)

In [0]:
describe detail flights_delays_delta

##### Temporay Views #####

Note that CTAS syntax doesn't support schema definition (it infers the schema from query results)

The recommended way to overcome it is to create a Temporay View, define or infer automatically schema and then run CTAS (similar as we did before but we used intermediate tables rather than temp views)

Temporary Viewes exist only during Spark Session.

In [0]:
 create
or replace temp view flights_delays_temp_view(
  year int,
  month int,
  day int,
  time int,
  timezone int,
  skycondition string,
  visibility string,
  weathertype string,
  drybulbfarenheit string,
  drybulbcelsius float,
  wetbulbfarenheit float,
  wetbulbcelsius float,
  dewpointfarenheit float,
  dewpointcelsius float,
  relativehumidity int,
  windspeed int,
  winddirection int,
  valueforwindcharacter int,
  stationpressure float,
  pressuretendency int,
  pressurechange int,
  sealevelpressure float,
  recordtype string,
  hourlyprecip string,
  altimeter float,
  airportcode string,
  displayairportname string,
  latitude float,
  longitude float
) using csv options (
  header = "true",
  path = "abfss://${container_name}@${storage_account}.dfs.core.windows.net/FlightsDelays/FlightWeatherWithAirportCode.csv"
)

Now run CTAS, which creates `managed` table from temp view. Why managed?

In [0]:
create
or replace table flights_delays_ctas as
select
  *
from
  flights_delays_temp_view

In [0]:
describe extended flights_delays_ctas

You can create a table in the specific location in Azure Blob Storage by using ***options ('path' 'abfss://${container_name}@${storage_account}.dfs.core.windows.net/new table location/')***

In this case an `external` table will be created.

In [0]:
create
or replace table flights_delays_with_path_option_ext
using delta options('path' 'abfss://${container_name}@${storage_account}.dfs.core.windows.net/FlightsDelays/bronze/${table_name}/')
as
select
  *
from
  flights_delays_temp_view

In [0]:
describe extended flights_delays_with_path_option_ext


##### Enriching data with additional meta-data

Databricks provides many built-in functions. In this example we use: `current_timestamp()` and `curent_user` functions.

There are many others, more details [databricks built-in functions](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/current_timestamp)

In [0]:
create
or replace table flights_delays_enriched as
select
  current_timestamp() as ingestiontime,
  current_user as user,
  *
from
  flights_delays_temp_view

In [0]:

select * from flights_delays_enriched limit 10